---
title: "Lab 1 · ¿Qué sabe realmente esta base sobre nosotros?"
subtitle: "Encuentro 1 · Unidad 1"
---

::: {.rds-meta}
[Laboratorio 1]{} [Encuentro 1]{} [≈ 35 minutos]{} [Python · pandas]{}
:::

Este cuaderno **no es una clase de programación**. El código está escrito, es
corto y a propósito poco sofisticado. Lo que se evalúa es la decisión que se
toma después de leer la salida.

Cada bloque termina en una **pregunta**. Respóndanla antes de ejecutar el
siguiente.

::: {.rds-sintetico}
`data/clientes_sinteticos.csv` es una base **sintética**, generada por
`scripts/generar_datos.py` con semilla fija. Ninguna persona real está
representada. Aun así, la vamos a tratar exactamente como trataríamos una base
real: esa es parte de la práctica.
:::

## 0 · Preparación

Una sola importación de terceros: `pandas`. `hashlib` viene con Python.

In [1]:
import hashlib
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 130)

# El cuaderno funciona tanto si se ejecuta desde labs/ como desde la raíz.
RUTA = Path("../data/clientes_sinteticos.csv")
if not RUTA.exists():
    RUTA = Path("data/clientes_sinteticos.csv")

print("Archivo:", RUTA.resolve().name)

Archivo: clientes_sinteticos.csv


## 1 · Cargar la base

Primera decisión, antes de mirar nada: ¿qué esperamos encontrar?

In [2]:
df = pd.read_csv(RUTA)
df.head(3)

,cliente_id,nombre,email,telefono,documento,fecha_nacimiento,edad,sexo,ciudad,barrio,direccion,latitud,longitud,estrato,ocupacion,ingresos_mensuales,dispositivo,ip,pais_residencia,canal_adquisicion,origen_dato,fecha_autorizacion,consentimiento_marketing,visitas_web,productos_vistos,categoria_top,compras_6m,monto_compras,fecha_ultima_compra,compras_farmacia_6m,entrega_asistida,busquedas_maternidad,score_riesgo_interno,segmento,churn
0,DM-00865,Luis Castro,luis.castro806@mail.co,3361052437,SYN-92480821,1992-12-31,33,M,Barranquilla,Simon Bolivar,Carrera 7 # 58-18,10.97607,-74.80194,2.0,Pensionado/a,2460000.0,Android,135.169.205.166,Colombia,email_marketing,lista_comprada_tercero,NaN,0,12,36,moda,2,418000.0,2026-08-29,0,0,0,26,fiel,0
1,DM-00714,Paula Beltran,paula.beltran226@correo.com,3552777695,SYN-95884611,1974-01-01,52,F,Cartagena,Manga,Carrera 53 # 50-88,10.33459,-75.45909,1.0,Comerciante,1710000.0,iOS,192.175.234.41,Colombia,publicidad_pagada,registro_cuenta,2024-02-23,1,9,28,supermercado,4,778000.0,2026-05-11,1,0,0,48,estable,0
2,DM-00572,Alvaro Riveros,alvaro.riveros457@mail.co,3693537285,SYN-35266492,2000-12-31,25,F,Barranquilla,El Prado,Diagonal 42 # 18-58,10.95000,-74.77236,3.0,Comerciante,2950000.0,Android,57.79.150.169,Alemania,publicidad_pagada,programa_fidelizacion,2025-05-14,1,12,19,tecnologia,2,435000.0,2026-07-26,0,0,0,41,estable,0


::: {.rds-card .decide}
**Pregunta 1.** Mire las tres primeras filas. Sin contar todavía las columnas:
¿cuántas de las que alcanza a ver le permitirían, por sí solas, llamar por
teléfono a esa persona?
:::

## 2 · El tamaño del problema

`shape` es la primera medida de riesgo de un proyecto: cuántas personas y
cuántas cosas sabemos de cada una.

In [3]:
filas, columnas = df.shape
print(f"Registros : {filas}")
print(f"Variables : {columnas}")
print(f"Celdas    : {filas * columnas:,}".replace(",", "."))

Registros : 1512
Variables : 35
Celdas    : 52.920


::: {.rds-card .reflexiona}
**Pregunta 2.** El correo de la Dirección Comercial decía «son unos 1 500
registros». Salen más. ¿Qué explicaciones se le ocurren, y cuál de ellas
sería un problema de calidad?
:::

## 3 · Las 35 variables, una por una

Leer la lista completa en voz alta es un ejercicio incómodo y por eso es útil.

In [4]:
for i, col in enumerate(df.columns, start=1):
    print(f"{i:>2}. {col}")

 1. cliente_id
 2. nombre
 3. email
 4. telefono
 5. documento
 6. fecha_nacimiento
 7. edad
 8. sexo
 9. ciudad
10. barrio
11. direccion
12. latitud
13. longitud
14. estrato
15. ocupacion
16. ingresos_mensuales
17. dispositivo
18. ip
19. pais_residencia
20. canal_adquisicion
21. origen_dato
22. fecha_autorizacion
23. consentimiento_marketing
24. visitas_web
25. productos_vistos
26. categoria_top
27. compras_6m
28. monto_compras
29. fecha_ultima_compra
30. compras_farmacia_6m
31. entrega_asistida
32. busquedas_maternidad
33. score_riesgo_interno
34. segmento
35. churn


## 4 · Tipos de datos

El tipo que `pandas` infiere **no** dice nada sobre el riesgo. `object` puede
ser un nombre propio o una categoría inofensiva; `int64` puede ser un conteo
de visitas o una edad.

In [5]:
resumen = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "no_nulos": df.notna().sum(),
    "distintos": df.nunique(),
})
resumen["% distintos"] = (resumen["distintos"] / len(df) * 100).round(1)
resumen.sort_values("% distintos", ascending=False).head(12)

,tipo,no_nulos,distintos,% distintos
cliente_id,str,1512,1512,100.0
email,str,1512,1500,99.2
telefono,str,1512,1500,99.2
direccion,str,1512,1500,99.2
documento,str,1512,1500,99.2
ip,str,1512,1500,99.2
latitud,float64,1512,1488,98.4
longitud,float64,1512,1483,98.1
fecha_autorizacion,str,1276,867,57.3
nombre,str,1512,769,50.9


::: {.rds-card .decide}
**Pregunta 3.** Las variables con un porcentaje de valores distintos cercano
al 100 % son casi siempre identificadores. ¿Cuáles aparecen en el top y cuál
de ellas *no* esperaba encontrar ahí?
:::

## 5 · Valores faltantes

La pregunta útil no es *cuántos* faltan, sino *por qué* faltan. Un faltante
puede ser un error de captura o la huella de un problema de origen.

In [6]:
faltantes = (
    df.isna().sum()
    .loc[lambda s: s > 0]
    .sort_values(ascending=False)
    .to_frame("faltantes")
)
faltantes["%"] = (faltantes["faltantes"] / len(df) * 100).round(1)
faltantes

,faltantes,%
fecha_autorizacion,236,15.6
ingresos_mensuales,135,8.9
ocupacion,76,5.0
estrato,60,4.0


In [7]:
# ¿Los faltantes de fecha_autorizacion están repartidos al azar?
tabla = pd.crosstab(
    df["origen_dato"],
    df["fecha_autorizacion"].isna().map({True: "sin fecha", False: "con fecha"}),
)
tabla

fecha_autorizacion,con fecha,sin fecha
origen_dato,,
enriquecimiento_web,0,92
entrega_domicilio,348,0
facturacion,464,0
lista_comprada_tercero,0,144
programa_fidelizacion,175,0
registro_cuenta,289,0


::: {.rds-card .riesgo}
**Pregunta 4.** Los faltantes de `fecha_autorizacion` no están repartidos al
azar: se concentran en dos orígenes. ¿Cuáles? ¿Y qué significa, en términos
prácticos, no tener fecha de autorización para esos registros?

Esto no es un problema de imputación. Es un problema de **evidencia**.
:::

## 6 · Clasificar las variables

Aquí empieza el trabajo real del laboratorio. Vamos a etiquetar cada una de
las 35 variables según el riesgo que introduce, no según su tipo.

Las categorías son las del Encuentro 1:

| Etiqueta | Significa |
|:--|:--|
| `directo` | Identifica a la persona por sí solo |
| `indirecto` | Identifica vía dispositivo o cuenta |
| `cuasi` | Solo no identifica; combinado con otros, sí |
| `comportamiento` | Lo que la persona hizo |
| `inferido` | Lo que la empresa dedujo |
| `proxy_sensible` | No es sensible, pero permite inferir uno |
| `control` | Metadato de tratamiento (origen, autorización) |
| `objetivo` | La variable que queremos predecir |

In [8]:
clasificacion = {
    # --- identificadores directos -------------------------------------
    "nombre": "directo", "email": "directo", "telefono": "directo",
    "documento": "directo", "direccion": "directo",
    # --- identificadores indirectos -----------------------------------
    "cliente_id": "indirecto", "ip": "indirecto", "dispositivo": "indirecto",
    "latitud": "indirecto", "longitud": "indirecto",
    # --- cuasi-identificadores ----------------------------------------
    "fecha_nacimiento": "cuasi", "edad": "cuasi", "sexo": "cuasi",
    "ciudad": "cuasi", "barrio": "cuasi", "ocupacion": "cuasi",
    "estrato": "cuasi", "ingresos_mensuales": "cuasi",
    "pais_residencia": "cuasi",
    # --- comportamiento ------------------------------------------------
    "visitas_web": "comportamiento", "productos_vistos": "comportamiento",
    "categoria_top": "comportamiento", "compras_6m": "comportamiento",
    "monto_compras": "comportamiento", "fecha_ultima_compra": "comportamiento",
    "canal_adquisicion": "comportamiento",
    # --- proxies de datos sensibles ------------------------------------
    "compras_farmacia_6m": "proxy_sensible",
    "entrega_asistida": "proxy_sensible",
    "busquedas_maternidad": "proxy_sensible",
    # --- datos inferidos por la empresa --------------------------------
    "score_riesgo_interno": "inferido", "segmento": "inferido",
    # --- metadatos de tratamiento --------------------------------------
    "origen_dato": "control", "fecha_autorizacion": "control",
    "consentimiento_marketing": "control",
    # --- objetivo -------------------------------------------------------
    "churn": "objetivo",
}

mapa = pd.Series(clasificacion, name="categoria").rename_axis("variable")
print("Variables clasificadas:", len(mapa), "de", df.shape[1])
mapa.value_counts().to_frame("n_variables")

Variables clasificadas: 35 de 35


,n_variables
categoria,
cuasi,9
comportamiento,7
directo,5
indirecto,5
proxy_sensible,3
control,3
inferido,2
objetivo,1


::: {.rds-card .decide}
**Pregunta 5.** Esta clasificación es **discutible a propósito**. Elija dos
variables que usted habría puesto en otra categoría y defienda el cambio.

Candidatas frecuentes: `latitud`/`longitud` (¿indirecto o cuasi?),
`ingresos_mensuales` (¿cuasi o proxy sensible?), `categoria_top`
(¿comportamiento o proxy sensible, si la categoría es `salud_bienestar`?).
:::

## 7 · Identificadores directos: qué tan directos son

Contamos cuántas personas quedan identificadas de forma única por cada
identificador directo.

In [9]:
directos = mapa[mapa == "directo"].index.tolist()

for col in directos:
    unicos = df[col].nunique(dropna=True)
    print(f"{col:<18} valores distintos: {unicos:>5}   "
          f"({unicos / len(df) * 100:.1f} % de los registros)")

nombre             valores distintos:   769   (50.9 % de los registros)
email              valores distintos:  1500   (99.2 % de los registros)
telefono           valores distintos:  1500   (99.2 % de los registros)
documento          valores distintos:  1500   (99.2 % de los registros)
direccion          valores distintos:  1500   (99.2 % de los registros)


::: {.rds-card .riesgo}
**Pregunta 6.** `email` tiene menos valores distintos que registros. Eso
significa que hay correos repetidos. ¿Son personas distintas con el mismo
correo, o la misma persona cargada dos veces? ¿Cómo lo comprobaría?
:::

In [10]:
# Comprobación
repetidos = df[df.duplicated("email", keep=False)].sort_values("email")
print("Registros con correo repetido:", len(repetidos))
repetidos[["cliente_id", "nombre", "email", "ciudad", "compras_6m"]].head(8)

Registros con correo repetido: 24


,cliente_id,nombre,email,ciudad,compras_6m
1441,DM-01442,Alvaro Mendoza,alvaro.mendoza285@webmail.net,Cartagena,6
1192,DM-90008,Alvaro Mendoza,alvaro.mendoza285@webmail.net,Cartagena,6
741,DM-90002,Ana Trujillo,ana.trujillo365@mail.co,Pasto,3
382,DM-00462,Ana Trujillo,ana.trujillo365@mail.co,Pasto,3
283,DM-00376,Camila Naranjo,camila.naranjo960@webmail.net,Medellin,4
299,DM-90005,Camila Naranjo,camila.naranjo960@webmail.net,Medellin,4
418,DM-90004,Claudia Barrios,claudia.barrios668@webmail.net,Villavicencio,2
257,DM-00728,Claudia Barrios,claudia.barrios668@webmail.net,Villavicencio,2


## 8 · Cuasi-identificadores: el riesgo que no se ve

Ninguna de estas columnas identifica sola. Vamos a ver qué pasa cuando se
combinan.

In [11]:
def riesgo_unicidad(datos: pd.DataFrame, columnas: list[str]) -> dict:
    # Cuenta cuántos registros quedan SOLOS en su grupo (k = 1).
    grupos = datos.groupby(columnas, dropna=False).size()
    unicos = int((grupos == 1).sum())
    return {
        "variables": " + ".join(columnas),
        "combinaciones": int(len(grupos)),
        "registros k=1": unicos,
        "% en riesgo": round(unicos / len(datos) * 100, 1),
    }


combinaciones = [
    ["ciudad"],
    ["ciudad", "sexo"],
    ["ciudad", "sexo", "edad"],
    ["ciudad", "sexo", "edad", "ocupacion"],
    ["ciudad", "barrio", "sexo", "edad"],
    ["ciudad", "barrio", "sexo", "edad", "ocupacion", "estrato"],
]

pd.DataFrame([riesgo_unicidad(df, c) for c in combinaciones])

,variables,combinaciones,registros k=1,% en riesgo
0,ciudad,12,0,0.0
1,ciudad + sexo,46,3,0.2
2,ciudad + sexo + edad,738,385,25.5
3,ciudad + sexo + edad + ocupacion,1385,1269,83.9
4,ciudad + barrio + sexo + edad,1194,945,62.5
5,ciudad + barrio + sexo + edad + ocupacion + es...,1496,1480,97.9


::: {.rds-card .riesgo}
**Pregunta 7.** Con una sola variable el riesgo es prácticamente cero. ¿A
partir de cuántas variables la mayoría de las personas de esta base queda
sola en su grupo?

Anote el número. Es el argumento que va a necesitar la próxima vez que
alguien diga «ya le quitamos los nombres».
:::

## 9 · Variables innecesarias para el problema

El encargo es predecir `churn`. Preguntémosle a la base qué variables
realmente se mueven con el abandono — **sin usar todavía ningún modelo**.

In [12]:
numericas = df.select_dtypes("number").columns.drop("churn")
correlaciones = (
    df[numericas].corrwith(df["churn"])
    .abs()
    .sort_values(ascending=False)
    .to_frame("|correlación| con churn")
    .round(3)
)
correlaciones["categoría"] = mapa.reindex(correlaciones.index).values
correlaciones

,|correlación| con churn,categoría
score_riesgo_interno,0.382,inferido
compras_6m,0.366,comportamiento
monto_compras,0.285,comportamiento
edad,0.185,cuasi
visitas_web,0.169,comportamiento
estrato,0.168,cuasi
compras_farmacia_6m,0.117,proxy_sensible
ingresos_mensuales,0.111,cuasi
entrega_asistida,0.099,proxy_sensible
busquedas_maternidad,0.090,proxy_sensible


::: {.rds-card .dilema}
**Pregunta 8.** Mire la columna `categoría` de la tabla anterior. Algunas de
las variables más correlacionadas con `churn` están etiquetadas como
`proxy_sensible` o `cuasi`.

¿Qué hace con ellas? Tres respuestas son defendibles:

1. Usarlas: correlacionan, el negocio lo pidió, el modelo mejora.
2. Excluirlas: el riesgo sobre las personas supera la ganancia predictiva.
3. Usarlas **condicionadas**: solo si existe una finalidad documentada, una
   base legal clara y un control sobre cómo se usa la predicción.

Elija una y escriba, en una frase, qué le diría a la Dirección Comercial.
:::

## 10 · Construir el dataset minimizado

Minimizar no es «quitar columnas». Es responder, para cada columna, *por qué
la necesito para esta finalidad concreta*.

In [13]:
# Finalidad declarada: predecir abandono para una campaña de retención.
variables_minimas = [
    "cliente_id",            # necesario para actuar sobre el cliente
    "visitas_web",
    "productos_vistos",
    "compras_6m",
    "monto_compras",
    "fecha_ultima_compra",
    "canal_adquisicion",
    "churn",
]

df_min = df[variables_minimas].copy()

print(f"Base original  : {df.shape[1]} variables")
print(f"Base minimizada: {df_min.shape[1]} variables")
print(f"Reducción      : {100 - df_min.shape[1] / df.shape[1] * 100:.0f} %")
print()
print("Variables eliminadas por categoría:")
eliminadas = mapa.drop(index=[v for v in variables_minimas if v in mapa.index])
print(eliminadas.value_counts().to_string())

Base original  : 35 variables
Base minimizada: 8 variables
Reducción      : 77 %

Variables eliminadas por categoría:
categoria
cuasi             9
directo           5
indirecto         4
proxy_sensible    3
control           3
inferido          2
comportamiento    1


::: {.rds-card .decide}
**Pregunta 9.** `cliente_id` se conservó. ¿Por qué? ¿Y qué habría que hacer
distinto si esta base fuera a salir de la empresa —por ejemplo, hacia un
proveedor de analítica?
:::

## 11 · Seudonimización

Reemplazamos el identificador por un hash. Es una buena práctica **y no es
anonimización**: vamos a comprobarlo en el bloque siguiente.

In [14]:
SAL = "datamarket-2026-curso"   # en producción: secreto, rotado y fuera del código


def seudonimizar(valor: str, sal: str = SAL, largo: int = 12) -> str:
    return hashlib.sha256(f"{sal}{valor}".encode("utf-8")).hexdigest()[:largo].upper()


df_seudo = df_min.copy()
df_seudo["cliente_id"] = df_seudo["cliente_id"].map(seudonimizar)

df_seudo.head(4)

,cliente_id,visitas_web,productos_vistos,compras_6m,monto_compras,fecha_ultima_compra,canal_adquisicion,churn
0,25D7EDAE7F3B,12,36,2,418000.0,2026-08-29,email_marketing,0
1,01A597CF34C8,9,28,4,778000.0,2026-05-11,publicidad_pagada,0
2,EAE0A059CED6,12,19,2,435000.0,2026-07-26,publicidad_pagada,0
3,7CFE67153C83,15,28,5,2586000.0,2026-03-30,marketplace,0


In [15]:
# La tabla de equivalencias: esto es lo que hace que NO sea anonimización.
tabla_equivalencias = pd.DataFrame({
    "cliente_id": df["cliente_id"],
    "seudonimo": df["cliente_id"].map(seudonimizar),
    "nombre": df["nombre"],
    "email": df["email"],
})
tabla_equivalencias.head(4)

,cliente_id,seudonimo,nombre,email
0,DM-00865,25D7EDAE7F3B,Luis Castro,luis.castro806@mail.co
1,DM-00714,01A597CF34C8,Paula Beltran,paula.beltran226@correo.com
2,DM-00572,EAE0A059CED6,Alvaro Riveros,alvaro.riveros457@mail.co
3,DM-00737,7CFE67153C83,Julian Rojas,julian.rojas77@mail.co


::: {.rds-card .riesgo}
**Pregunta 10.** Mientras exista la tabla anterior, los datos siguen siendo
datos personales.

Y la tabla **tiene que existir**, porque sin ella la empresa no puede llamar
al cliente que el modelo señaló. Ese es el nudo: la finalidad del proyecto
—actuar sobre personas concretas— es incompatible con la anonimización real.

¿Dónde debería vivir esa tabla, y quién debería poder leerla?
:::

## 12 · La prueba definitiva: eliminar el nombre no anonimiza

Simulamos exactamente lo que hizo el equipo en el Caso B: borrar los
identificadores directos y quedarse con lo demás.

In [16]:
caso_b = df[["edad", "sexo", "ciudad", "ocupacion", "categoria_top",
             "compras_farmacia_6m"]].copy()

print("Columnas de la base 'depurada':", list(caso_b.columns))
print("¿Queda algún nombre, correo o documento?  ->  No")
print()

resultado = riesgo_unicidad(caso_b, ["edad", "sexo", "ciudad", "ocupacion"])
for k, v in resultado.items():
    print(f"{k:>16}: {v}")

Columnas de la base 'depurada': ['edad', 'sexo', 'ciudad', 'ocupacion', 'categoria_top', 'compras_farmacia_6m']
¿Queda algún nombre, correo o documento?  ->  No

       variables: edad + sexo + ciudad + ocupacion
   combinaciones: 1385
   registros k=1: 1269
     % en riesgo: 83.9


In [17]:
# ¿Quiénes son los casos más expuestos? Los grupos de tamaño 1.
grupos = caso_b.groupby(["edad", "sexo", "ciudad", "ocupacion"], dropna=False).size()
solos = grupos[grupos == 1]

print(f"Personas solas en su grupo: {len(solos)}")
print("\nAlgunos ejemplos (cada fila es UNA persona identificable por cruce):\n")
print(solos.head(10).to_string())

Personas solas en su grupo: 1269

Algunos ejemplos (cada fila es UNA persona identificable por cruce):

edad  sexo  ciudad        ocupacion    
18    F     Barranquilla  Abogado/a        1
                          Ingeniero/a      1
            Bogota        Contador/a       1
                          Disenador/a      1
                          NaN              1
            Bucaramanga   Abogado/a        1
                          Disenador/a      1
                          Pensionado/a     1
                          Tecnico/a        1
            Cali          Independiente    1


::: {.rds-card .decision}
**Pregunta 11.** Esas filas son personas. Si la base incluyera además un
diagnóstico, una orientación política o un historial crediticio, cualquiera
con acceso a un directorio profesional podría asociarlos a un nombre.

Escriba la decisión: ¿esta base se puede compartir con un tercero? Si su
respuesta es «sí, con condiciones», enumere las condiciones.
:::

## 13 · Reducir el riesgo y medirlo

Anonimizar es un resultado que se mide. Apliquemos tres transformaciones y
volvamos a medir.

In [18]:
caso_b_v2 = caso_b.copy()

# (a) Edad en rangos quinquenales
caso_b_v2["edad"] = pd.cut(caso_b_v2["edad"], bins=range(15, 90, 5)).astype(str)

# (b) Ocupación agrupada en tres grandes bloques
grandes = {
    "Ingeniero/a": "técnico-profesional", "Tecnico/a": "técnico-profesional",
    "Disenador/a": "técnico-profesional", "Contador/a": "técnico-profesional",
    "Abogado/a": "técnico-profesional", "Administrador/a": "técnico-profesional",
    "Docente": "servicios", "Enfermero/a": "servicios",
    "Comerciante": "servicios", "Independiente": "servicios",
    "Estudiante": "sin actividad remunerada",
    "Pensionado/a": "sin actividad remunerada",
}
caso_b_v2["ocupacion"] = caso_b_v2["ocupacion"].map(grandes).fillna("otro")

# (c) Ciudad -> región
regiones = {
    "Bogota": "Centro", "Ibague": "Centro", "Villavicencio": "Centro",
    "Medellin": "Antioquia-Eje", "Pereira": "Antioquia-Eje", "Manizales": "Antioquia-Eje",
    "Cali": "Pacífico", "Pasto": "Pacífico",
    "Barranquilla": "Caribe", "Cartagena": "Caribe", "Santa Marta": "Caribe",
    "Bucaramanga": "Nororiente",
}
caso_b_v2["ciudad"] = caso_b_v2["ciudad"].map(regiones).fillna("otro")

comparacion = pd.DataFrame([
    {"versión": "original", **riesgo_unicidad(caso_b, ["edad", "sexo", "ciudad", "ocupacion"])},
    {"versión": "generalizada", **riesgo_unicidad(caso_b_v2, ["edad", "sexo", "ciudad", "ocupacion"])},
])
comparacion

,versión,variables,combinaciones,registros k=1,% en riesgo
0,original,edad + sexo + ciudad + ocupacion,1385,1269,83.9
1,generalizada,edad + sexo + ciudad + ocupacion,406,160,10.6


::: {.rds-card .prueba}
**Pregunta 12.** El porcentaje en riesgo bajó. ¿Bajó lo suficiente?

No hay un umbral universal. Lo que sí hay es una obligación: **decir cuál es
el umbral que se aceptó y por qué**. Escriba el suyo.

Y la contrapartida: ¿qué análisis dejó de ser posible con la base
generalizada? ¿Vale la pena?
:::

## 14 · Cierre del laboratorio

Lo que hicimos, en orden:

1. Medimos el tamaño del problema.
2. Descubrimos que los faltantes de `fecha_autorizacion` no eran aleatorios:
   eran una huella del origen de los datos.
3. Clasificamos 35 variables por riesgo, no por tipo.
4. Comprobamos que cuatro cuasi-identificadores dejan sola a la gran mayoría
   de las personas de la base.
5. Construimos una base minimizada para una finalidad concreta.
6. Seudonimizamos y vimos por qué eso no anonimiza.
7. Generalizamos y **medimos** cuánto bajó el riesgo.

::: {.rds-card .reflexiona}
**Pregunta de salida.** Tome una base con la que trabaje realmente. Ejecute
mentalmente el bloque 8 sobre ella: ¿cuántas variables cuasi-identificadoras
tiene?

::: {.pregunta}
¿Qué tendría que cambiar en mi proyecto de datos?
:::
:::

En el [Laboratorio 2](lab02.ipynb) pasamos de la base al *pipeline* completo:
auditamos cada etapa y comparamos dos modelos de *churn* que difieren en AUC
y en lo que están dispuestos a usar.